# NER with CNN-BiLSTM

Giuseppe Termerissa


## Setup


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import os
import json
import numpy as np
import glob
import shutil
import tensorflow

import sys

sys.path.insert(0, "./src/")

2026-10-28 11:59:08.545561: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


### Download dependencies

This is unnecessary unless the notebook is being ran on Colab with no pre-downloaded datasets, and will be normally skipped.


In [3]:
try:
    import google.colab  # type: ignore

    IN_COLAB = True
except Exception:
    IN_COLAB = False

In [2]:
import os

if "glove.6B" not in os.listdir("../data"):
    !wget "https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip"

    # four file with different numbers of parameters: 50, 100, 200, 300
    !unzip glove.6B.zip -d ../data/glove.6B/

--2026-10-28 12:11:46--  https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip
Resolving downloads.cs.stanford.edu (downloads.cs.stanford.edu)... 171.64.64.22
Connecting to downloads.cs.stanford.edu (downloads.cs.stanford.edu)|171.64.64.22|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 862182613 (822M) [application/zip]
Saving to: ‘glove.6B.zip’

glove.6B.zip        100%[===================>] 822.24M  5.02MB/s    in 2m 41s  

2026-10-28 12:14:28 (5.12 MB/s) - ‘glove.6B.zip’ saved [862182613/862182613]

Archive:  glove.6B.zip
  inflating: ../data/glove.6B/glove.6B.50d.txt  
  inflating: ../data/glove.6B/glove.6B.100d.txt  
  inflating: ../data/glove.6B/glove.6B.200d.txt  
  inflating: ../data/glove.6B/glove.6B.300d.txt  


## Pipeline


In [5]:
hyper = {
    # CNN hyperameters
    "CONV_WIDTH": 4,
    "FILTERS": 30,
    # BiLSTM hyperparameters
    "LSTM_STATE_SIZE": 300,
    "LSTM_DROPOUT": 0.68,
    "REC_DROPOUT": 0.25,
    # Model hyperparameters
    "WORD_EMB_DIR": "glove.6B/glove.6B.100d.txt",
    "EPOCHS": 80,
    "BATCH_SIZE": 128,
    "OPTIMIZER": "rmsprop",
}

In [6]:
from src.datasets import load_conll2003

conll2012_dir = "../data/conll-2012"
conll2003_dir = "../data/conll003-englishversion"

train_sentences, test_sentences = load_conll2003(conll2003_dir)

len(train_sentences), len(test_sentences)

(17291, 3453)

In [7]:
from src.pipeline import (
    unpack_sentences,
    build_part_of_phrase_lu,
    build_labels_lu,
    caps_lu,
    load_word_embeddings,
    create_char_lu,
)

labels, vocab, phrase_parts = unpack_sentences([train_sentences, test_sentences])

phrase_lu, phrase_emb = build_part_of_phrase_lu(phrase_parts)

labels_lu = build_labels_lu(labels)

case_emb = np.identity(len(caps_lu), dtype="float32")

words_lu, word_emb = load_word_embeddings(f"../data/{hyper['WORD_EMB_DIR']}", vocab)

char_lu = create_char_lu()

In [8]:
from src.pipeline import sentences_to_matrix


train_set = sentences_to_matrix(
    train_sentences, caps_lu, words_lu, char_lu, phrase_lu, labels_lu
)
test_set = sentences_to_matrix(
    test_sentences, caps_lu, words_lu, char_lu, phrase_lu, labels_lu
)

## Model Setup


In [9]:
from src.model import create_model


model = create_model(
    hyper,
    word_emb,
    case_emb,
    phrase_emb,
    char_lu,
    labels_lu,
)

I0000 00:00:1793185154.617214   28806 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5562 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3070, pci bus id: 0000:26:00.0, compute capability: 8.6


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ char_input          │ (None, None, 52)  │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ char_embedding      │ (None, None, 52,  │      3,060 │ char_input[0][0]  │
│ (TimeDistributed)   │ 30)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, None, 52,  │          0 │ char_embedding[0… │
│                     │ 30)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed    │ (None, None, 52,  │      3,630 │ dropout[0][0]     │
│ (TimeDistributed)   │ 30)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed_1  │ (None, None, 1,   │          0 │ time_distributed… │
│ (TimeDistributed)   │ 30)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ words_input         │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ casing_input        │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed_2  │ (None, None, 30)  │          0 │ time_distributed… │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ phrasetag_input     │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, None, 100) │  2,294,900 │ words_input[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, None, 4)   │         16 │ casing_input[0][… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, None, 30)  │          0 │ time_distributed… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_2         │ (None, None, 22)  │        484 │ phrasetag_input[… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, None, 156) │          0 │ embedding[0][0],  │
│ (Concatenate)       │                   │            │ embedding_1[0][0… │
│                     │                   │            │ dropout_1[0][0],  │
│                     │                   │            │ embedding_2[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional       │ (None, None, 600) │  1,096,800 │ concatenate[0][0] │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed_3  │ (None, None, 10)  │      6,010 │ bidirectional[0]… │
│ (TimeDistributed)   │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴─────────────────

 Total params: 3,404,900 (12.99 MB)

 Trainable params: 1,109,500 (4.23 MB)

 Non-trainable params: 2,295,400 (8.76 MB)

## Training


In [ ]:
from src.pipeline import pad_dataset

tokens, caps, chars, labels, phrasetags = pad_dataset(train_set)

X = [tokens, caps, chars, phrasetags]
y = labels

history = model.fit(
    X,
    y,
    epochs=hyper["EPOCHS"],
    batch_size=hyper["BATCH_SIZE"],
)

Epoch 1/80


2026-10-28 11:59:16.392805: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 187019456 exceeds 10% of free system memory.
2026-10-28 11:59:29.100852: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 92700


136/136 ━━━━━━━━━━━━━━━━━━━━ 139s 920ms/step - loss: 0.1983
Epoch 2/80
136/136 ━━━━━━━━━━━━━━━━━━━━ 123s 906ms/step - loss: 0.1009
Epoch 3/80
 17/136 ━━━━━━━━━━━━━━━━━━━━ 1:48 908ms/step - loss: 0.0867

## Evaluation


In [ ]:
# compute predictions on the test set
from src.evaluation import predict_dataset

y_true, y_pred = predict_dataset(test_set, model)

# reverse label lookup
label_idx_lu = {v: k for k, v in labels_lu.items()}

# convert back to labels
pred_label = []
for sentence in y_pred:
    pred_label.append([label_idx_lu[element] for element in sentence])

true_label = []
for sentence in y_true:
    true_label.append([label_idx_lu[element] for element in sentence])

In [ ]:
from src.evaluation import evaluate_pred

pre, rec, f1 = evaluate_pred(pred_label, true_label)

print(f"Precision:  {pre:.3f}")
print(f"Recall:     {rec:.3f}")
print(f"F1-Score:   {f1:.3f}")